In [34]:
from kafka import KafkaConsumer
from hw_models import Ride, ride_deserializer

server = 'localhost:9092'
topic_name = 'green-trips'



consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='latest',
    group_id='green-trips-v3',
    value_deserializer=ride_deserializer
)

In [35]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)

conn.autocommit = True
cur = conn.cursor()

In [ ]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
try:
    for message in consumer:
        ride = message.value
        
        # Convert the string to a float/int first, then divide
        pickup_dt = ride.lpep_pickup_datetime
        dropoff_dt = ride.lpep_dropoff_datetime
        # pickup_dt = datetime.fromtimestamp(float(ride.lpep_pickup_datetime) / 1000)
        # dropoff_dt = datetime.fromtimestamp(float(ride.lpep_dropoff_datetime) / 1000)

        cur.execute(
            """INSERT INTO green_trips (
                   lpep_pickup_datetime, 
                   lpep_dropoff_datetime, 
                   PULocationID, 
                   DOLocationID, 
                   passenger_count, 
                   trip_distance, 
                   tip_amount, 
                   total_amount
               )
               VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
            (
                pickup_dt, 
                dropoff_dt, 
                ride.PULocationID, 
                ride.DOLocationID, 
                ride.passenger_count, 
                ride.trip_distance, 
                ride.tip_amount, 
                ride.total_amount
            )
        )
        
        count += 1
        
        # CRITICAL: Commit the transaction so data is saved
        if count % 100 == 0:
            conn.commit()
            print(f"Inserted and committed {count} rows...")

except KeyboardInterrupt:
    print("Stopping consumer...")

finally:
    # Final commit for remaining rows
    consumer.close()
    cur.close()
    conn.close()